# Shared ML Core Weeks 1–4 Student TODO and Debugging Notebook

Complete every numbered TODO and repair both debugging checkpoints in order. The notebook contains exactly **two intentional ML-workflow bugs**; TODO placeholders are unfinished student work, not additional hidden bugs.

Keep the official test set disabled until every model choice is frozen and your instructor approves the final evaluation. Base every observation and conclusion on evidence from your own runs.


## Environment and reproducibility

Target Python 3.10 or newer. For Colab, install `ogb`, `torch-geometric`, `scikit-learn`, `pandas`, and `matplotlib`. For a local environment, install the same packages with Conda or pip.

The notebook supports the full official OGBN-ArXiv partitions on CPU. Random seeds and experiment settings are centralized in `CONFIG`.

**TODO SETUP.1:** Run the version-report cell and record any environment changes needed to reproduce your work.

**Course AI-use policy:** students should not use autonomous coding agents to plan, implement, run, or complete this project. Limited use for conceptual explanations, documentation lookup, and debugging assistance is acceptable.


In [1]:
from __future__ import annotations

import copy
import platform
import random
import time
from importlib import metadata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from ogb.nodeproppred import PygNodePropPredDataset
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

CONFIG = {
    "seed": 42,
    "data_root": "./data",
    "pca_fit_points": 30_000,
    "pca_plot_points": 10_000,
    "batch_size": 4_096,
    "epochs": 30,
    "hidden_dim": 128,
    "logistic_c_values": [0.1, 1.0, 10.0],
    "logistic_max_iter": 120,
    "logistic_tolerance": 1e-3,
    "learning_curve_sizes": [5_000, 20_000, 50_000, "full"],
    "calibration_bins": 10,
    "run_final_test": False,
    "model_selection_frozen": False,
}

SEED = CONFIG["seed"]
RUN_FINAL_TEST = CONFIG["run_final_test"]
MODEL_SELECTION_FROZEN = CONFIG["model_selection_frozen"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Run mode: full official splits")
print("Device:", DEVICE)


ModuleNotFoundError: No module named 'matplotlib'

In [2]:
def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"


version_report = pd.Series({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "torch": torch.__version__,
    "ogb": package_version("ogb"),
    "torch-geometric": package_version("torch-geometric"),
}, name="version")
version_report


NameError: name 'pd' is not defined

In [3]:
def set_seed(seed: int = 42) -> None:
    '''Set all random generators used by this notebook.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


NameError: name 'SEED' is not defined

# Week 1 Dataset Splits EDA and Preprocessing

Establish a correct, reproducible, leakage-safe workflow before fitting any classifier.


## TODO W1.1 Explain the official split

In your own words, write one or two sentences for each prompt:

- What does the training set control?
- What is the validation set used for?
- When may the test set be evaluated?
- Why does repeatedly checking test performance leak information into model development?

**Your response:**

- Training:
- Validation:
- Test:
- Leakage risk:


In [ ]:
def load_ogbn_arxiv(data_root: str | Path):
    '''Instructor-provided loader for OGBN-ArXiv and its official time-based split.'''
    dataset = PygNodePropPredDataset(name="ogbn-arxiv", root=str(data_root))
    data = dataset[0]
    return dataset, data, dataset.get_idx_split()


def split_report(split_idx: dict[str, torch.Tensor]) -> dict[str, int]:
    '''TODO W1.2a: return the number of indices in each official partition.'''
    raise NotImplementedError("TODO W1.2a: summarize the official partitions")


def assert_official_splits_are_disjoint(
    split_idx: dict[str, torch.Tensor], num_nodes: int
) -> None:
    '''TODO W1.2b: assert pairwise disjointness and complete node coverage.'''
    raise NotImplementedError("TODO W1.2b: verify the official split")


In [ ]:
dataset, data, split_idx = load_ogbn_arxiv(CONFIG["data_root"])
x = data.x.float()
y = data.y.squeeze(-1).long()
edge_index = data.edge_index.long()

train_idx = split_idx["train"]
valid_idx = split_idx["valid"]
test_idx = split_idx["test"]

assert x.ndim == 2 and x.shape[1] == 128
assert y.ndim == 1 and y.shape[0] == x.shape[0]
assert int(y.min()) == 0 and int(y.max()) == 39
assert set(split_idx) == {"train", "valid", "test"}
assert_official_splits_are_disjoint(split_idx, x.shape[0])

# TODO W1.2c: create a pandas Series containing the node, edge, feature,
# class, and official partition counts, then display it.
dataset_summary = None
dataset_summary


In [ ]:
def stratified_subset(
    indices: torch.Tensor,
    labels: torch.Tensor,
    size: int,
    seed: int = 42,
) -> torch.Tensor:
    '''Select a reproducible subset strictly within an existing partition.'''
    if size >= int(indices.numel()):
        return indices.clone()
    index_np = indices.cpu().numpy()
    selected, _ = train_test_split(
        index_np,
        train_size=size,
        random_state=seed,
        stratify=labels[indices].cpu().numpy(),
    )
    return torch.as_tensor(selected, dtype=torch.long)


In [ ]:
def class_counts(labels: torch.Tensor, indices: torch.Tensor, n_classes: int = 40):
    '''TODO W1.3a: return the count for every class, including absent classes.'''
    raise NotImplementedError("TODO W1.3a: calculate class counts")


# TODO W1.3b:
# 1. Calculate training and validation counts for all 40 classes.
# 2. Convert counts to proportions within each partition.
# 3. Build a tidy DataFrame named class_distribution.
# 4. Plot the training and validation class proportions together.
raise NotImplementedError("TODO W1.3b: create the class-distribution analysis")


In [ ]:
pca_fit_idx = stratified_subset(
    train_idx, y, min(CONFIG["pca_fit_points"], len(train_idx)), SEED
)
pca_plot_idx = stratified_subset(
    train_idx, y, min(CONFIG["pca_plot_points"], len(train_idx)), SEED + 1
)

# TODO W1.4:
# 1. Create a two-component PCA object.
# 2. Fit it using pca_fit_idx only; these rows must belong to the training split.
# 3. Transform pca_plot_idx.
# 4. Make a labeled scatter plot and report explained variance.
raise NotImplementedError("TODO W1.4: fit and plot training-only PCA")


In [ ]:
def graph_degrees(edge_index: torch.Tensor, num_nodes: int):
    '''TODO W1.5a: calculate in-degree and out-degree for every node.'''
    raise NotImplementedError("TODO W1.5a: calculate graph degrees")


# TODO W1.5b:
# 1. Summarize in-degree and out-degree with useful percentiles.
# 2. Plot both degree distributions. A log1p x-axis transformation is helpful.
# 3. Label the axes and legend clearly.
raise NotImplementedError("TODO W1.5b: summarize and plot graph degrees")


In [ ]:
# TODO W1.6:
# Inspect feature scale using training rows only. At minimum, summarize each
# feature's mean, standard deviation, minimum, and maximum. Add one plot that
# helps you decide whether standardization is reasonable.
raw_feature_summary = None
raise NotImplementedError("TODO W1.6: inspect training-feature scale")


### Debugging checkpoint 1 Learned preprocessing

**Expected invariant:** any transformation that learns statistics must fit on official training rows only. The next cell still executes, so verify the data flow and the reported fit count rather than waiting for an exception. Repair this checkpoint before using the preprocessed features in later weeks.


In [ ]:
def fit_preprocessor(features: torch.Tensor, train_indices: torch.Tensor) -> StandardScaler:
    '''Fit a StandardScaler for use across the official partitions.'''
    features_np = features.cpu().numpy()
    scaler = StandardScaler()
    scaler.fit(features_np)
    return scaler


def transform_features(features: torch.Tensor, scaler: StandardScaler) -> np.ndarray:
    return scaler.transform(features.cpu().numpy()).astype(np.float32, copy=False)


scaler = fit_preprocessor(x, train_idx)
x_scaled = transform_features(x, scaler)
standardized_train = x_scaled[train_idx.numpy()]

preprocessing_check = pd.Series({
    "rows_seen_during_fit": int(scaler.n_samples_seen_),
    "official_training_rows": len(train_idx),
    "mean_absolute_feature_mean": float(np.abs(standardized_train.mean(axis=0)).mean()),
    "mean_feature_std": float(standardized_train.std(axis=0).mean()),
}, name="value")
preprocessing_check


## TODO W1.7 Week 1 observations

Write **3–5 evidence-based observations** using your tables and plots. Include at least one observation about class imbalance, one about the feature or PCA view, and one about graph degree. Then explain why learned preprocessing must be fitted without validation or test rows.

1.
2.
3.
4. *(optional)*
5. *(optional)*

**Why train-only fitting matters:**


# Week 2 Baselines Logistic Regression Metrics and Error Analysis

Report Accuracy and Macro-F1 together. Select hyperparameters using validation performance only and record every attempted configuration.


In [ ]:
def score_predictions(y_true, y_pred) -> dict[str, float]:
    '''TODO W2.1a: accept NumPy arrays or CPU torch tensors and return a dict with exactly the keys `accuracy` and `macro_f1`; both values must be floats.'''
    raise NotImplementedError("TODO W2.1a: implement the metric helper")


def majority_class_predictions(training_labels, output_size: int):
    '''TODO W2.1b: learn the majority class from training labels only.'''
    raise NotImplementedError("TODO W2.1b: implement the trivial baseline")


def record_experiment(records: list[dict], **result) -> None:
    '''Instructor-provided helper: keep every result, not only the best one.'''
    records.append(dict(result))


y_np = y.cpu().numpy()
train_np = train_idx.cpu().numpy()
valid_np = valid_idx.cpu().numpy()
experiment_records = []

# TODO W2.1c: generate majority-class predictions for train and validation,
# score both partitions, record the experiment, and display the record table.
raise NotImplementedError("TODO W2.1c: evaluate and record the trivial baseline")


In [ ]:
def make_logistic_model(c_value: float):
    '''TODO W2.2a: return a multinomial-capable LogisticRegression model.'''
    raise NotImplementedError("TODO W2.2a: configure Logistic Regression")


def run_logistic_search(
    features: np.ndarray,
    labels: np.ndarray,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
    c_values,
):
    '''TODO W2.2b: return `(best_model, best_c, results_df)` after selecting with validation Macro-F1 only. `results_df` must contain `C`, `train_accuracy`, `val_accuracy`, `train_macro_f1`, and `val_macro_f1`.'''
    # For every C value:
    # - fit on train_indices;
    # - calculate train and validation Accuracy and Macro-F1;
    # - save the fitted model and a result row.
    # Return (best_model, best_c, results_df). The result table must use the
    # exact column names stated in the docstring because later cells read them.
    # This function intentionally has no test-index argument.
    raise NotImplementedError("TODO W2.2b: implement validation-only model selection")


In [ ]:
best_logistic_model, best_logistic_c, logistic_results = run_logistic_search(
    x_scaled,
    y_np,
    train_np,
    valid_np,
    CONFIG["logistic_c_values"],
)

for row in logistic_results.to_dict("records"):
    record_experiment(
        experiment_records,
        model="Logistic Regression",
        configuration=f"C={row['C']:g}",
        train_accuracy=row["train_accuracy"],
        val_accuracy=row["val_accuracy"],
        train_macro_f1=row["train_macro_f1"],
        val_macro_f1=row["val_macro_f1"],
    )

print(f"Locked Logistic Regression C={best_logistic_c:g} using validation Macro-F1.")
logistic_results


In [ ]:
# TODO W2.3: Build a Logistic Regression learning curve.
# - Use CONFIG["learning_curve_sizes"].
# - Draw deterministic stratified subsets strictly within train_idx.
# - Fit the locked C for each subset.
# - Record train/validation Accuracy and Macro-F1.
# - Plot both metrics against training-set size.
learning_curve_rows = []
logistic_learning_curve = None
raise NotImplementedError("TODO W2.3: create the Logistic Regression learning curve")


In [ ]:
logistic_train_pred = best_logistic_model.predict(x_scaled[train_np])
logistic_valid_pred = best_logistic_model.predict(x_scaled[valid_np])

# TODO W2.4:
# 1. Calculate train and validation metrics.
# 2. Compute and plot the 40-class validation confusion matrix.
# 3. Create class_performance with precision, recall, F1, and support.
# 4. Identify weak classes and the most common off-diagonal confusion.
logistic_train_metrics = None
logistic_valid_metrics = None
class_performance = None
raise NotImplementedError("TODO W2.4: perform class-level evaluation")


## TODO W2.5 Generalization metrics and error analysis

Answer with evidence from your results:

- What is the train–validation gap? Does it suggest overfitting, underfitting, or a reasonable fit?
- Does increasing the amount of training data improve validation performance?
- Why can Accuracy and Macro-F1 tell different stories on this dataset?
- Which classes perform poorly, and are they rare?
- Which class pairs are commonly confused?

Write **3–5 short observations or hypotheses**. Clearly label speculation as a hypothesis.

1.
2.
3.
4. *(optional)*
5. *(optional)*


# Week 3 First Neural Network and the Training Loop

Build a `128 → 128 → 40` MLP, implement the training loop, track every required metric, and compare the initial network with locked Logistic Regression.


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=128, hidden_dim=128, n_classes=40, dropout=0.0):
        super().__init__()
        # TODO W3.1a: define Linear -> ReLU -> Dropout -> Linear.
        # The output must contain unnormalized logits with shape [batch, 40].
        self.network = None

    def forward(self, features):
        # TODO W3.1b: return the network's logits.
        raise NotImplementedError("TODO W3.1b: implement the forward pass")


def make_loader(features, labels, indices, batch_size, shuffle, seed=42):
    '''Instructor-provided deterministic minibatch loader.'''
    index_np = indices.cpu().numpy()
    dataset = TensorDataset(
        torch.from_numpy(features[index_np]).float(),
        labels[indices].long().cpu(),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
    )


## TODO W3.2 Implement one training epoch

Complete the next function using this sequence for every minibatch: move data to the device, clear old gradients, run the forward pass, compute cross-entropy loss, backpropagate, update parameters, and accumulate loss and predictions.


In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device=DEVICE):
    model.train()

    # TODO W3.2:
    # - initialize accumulators;
    # - loop over batches;
    # - call optimizer.zero_grad(...) before the forward/backward pass;
    # - calculate logits and loss;
    # - call loss.backward() and optimizer.step();
    # - return average loss, Accuracy, and Macro-F1.
    raise NotImplementedError("TODO W3.2: implement one training epoch")


### Debugging checkpoint 2 Evaluation behavior

**Expected invariant:** evaluation disables training-only behavior and gradient tracking. Check whether repeated predictions from a model containing dropout are stable and whether returned logits require gradients. Repair this checkpoint before running MLP experiments or selecting a final model.


In [ ]:
def evaluate_model(model, loader, loss_fn, device=DEVICE, return_logits=False):
    model.train()
    total_loss = 0.0
    total_examples = 0
    all_targets = []
    all_predictions = []
    all_logits = []

    for batch_features, batch_targets in loader:
        batch_features = batch_features.to(device)
        batch_targets = batch_targets.to(device)
        logits = model(batch_features)
        loss = loss_fn(logits, batch_targets)
        batch_size = batch_targets.shape[0]
        total_loss += float(loss) * batch_size
        total_examples += batch_size
        all_targets.append(batch_targets.cpu())
        all_predictions.append(logits.argmax(dim=1).cpu())
        if return_logits:
            all_logits.append(logits.cpu())

    targets = torch.cat(all_targets)
    predictions = torch.cat(all_predictions)
    result = {
        "loss": total_loss / total_examples,
        **score_predictions(targets, predictions),
        "targets": targets,
        "predictions": predictions,
    }
    if return_logits:
        result["logits"] = torch.cat(all_logits)
    return result


In [ ]:
def train_mlp_experiment(name, learning_rate, dropout, weight_decay, epochs=None):
    epochs = epochs or CONFIG["epochs"]
    set_seed(SEED)
    train_loader = make_loader(x_scaled, y, train_idx, CONFIG["batch_size"], True, SEED)
    train_eval_loader = make_loader(x_scaled, y, train_idx, CONFIG["batch_size"], False, SEED)
    valid_loader = make_loader(x_scaled, y, valid_idx, CONFIG["batch_size"], False, SEED)

    model = MLP(
        input_dim=x_scaled.shape[1],
        hidden_dim=CONFIG["hidden_dim"],
        n_classes=40,
        dropout=dropout,
    ).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    history = []
    best_state = None
    best_epoch = None
    best_validation_f1 = -np.inf
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        train_metrics = evaluate_model(model, train_eval_loader, loss_fn, DEVICE)
        validation_metrics = evaluate_model(model, valid_loader, loss_fn, DEVICE)
        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "val_loss": validation_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_accuracy": validation_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_macro_f1": validation_metrics["macro_f1"],
        })
        if validation_metrics["macro_f1"] > best_validation_f1:
            best_validation_f1 = validation_metrics["macro_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"{name} epoch {epoch:02d}/{epochs}: "
                f"train F1={train_metrics['macro_f1']:.4f}, val F1={validation_metrics['macro_f1']:.4f}"
            )

    model.load_state_dict(best_state)
    elapsed = time.perf_counter() - started
    final_train = evaluate_model(model, train_eval_loader, loss_fn, DEVICE)
    final_validation = evaluate_model(model, valid_loader, loss_fn, DEVICE, return_logits=True)
    return {
        "name": name,
        "model": model,
        "loss_fn": loss_fn,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "seconds": elapsed,
        "learning_rate": learning_rate,
        "dropout": dropout,
        "weight_decay": weight_decay,
        "train_metrics": final_train,
        "validation_metrics": final_validation,
    }


def plot_training_history(history: pd.DataFrame, title: str):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, train_col, val_col, metric in [
        (axes[0], "train_loss", "val_loss", "Loss"),
        (axes[1], "train_accuracy", "val_accuracy", "Accuracy"),
        (axes[2], "train_macro_f1", "val_macro_f1", "Macro-F1"),
    ]:
        ax.plot(history["epoch"], history[train_col], label="train")
        ax.plot(history["epoch"], history[val_col], label="validation")
        ax.set(title=metric, xlabel="Epoch")
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


In [ ]:
initial_mlp_run = train_mlp_experiment(
    name="initial_mlp",
    learning_rate=1e-3,
    dropout=0.0,
    weight_decay=0.0,
)
plot_training_history(initial_mlp_run["history"], "Initial MLP")

initial_mlp_summary = pd.Series({
    "best_epoch": initial_mlp_run["best_epoch"],
    "seconds": initial_mlp_run["seconds"],
    "train_accuracy": initial_mlp_run["train_metrics"]["accuracy"],
    "val_accuracy": initial_mlp_run["validation_metrics"]["accuracy"],
    "train_macro_f1": initial_mlp_run["train_metrics"]["macro_f1"],
    "val_macro_f1": initial_mlp_run["validation_metrics"]["macro_f1"],
}, name="initial_mlp")
initial_mlp_summary


In [ ]:
# TODO W3.3: Create a two-row comparison between the locked Logistic
# Regression model and the initial MLP. Include train/validation Accuracy,
# train/validation Macro-F1, and each model's train-validation gap.
week3_first_comparison = None
raise NotImplementedError("TODO W3.3: compare the initial MLP with Logistic Regression")


## TODO W3.4 Explain the training loop

In your own words, explain:

- what the forward pass produces;
- what cross-entropy loss measures;
- why gradients are cleared before each new batch;
- what `loss.backward()` and `optimizer.step()` do;
- why low training loss alone does not prove good generalization;
- how training mode differs from evaluation mode.

**Your response:**


# Week 4 Generalization Regularization and Model Comparison

Diagnose the initial MLP, change one factor at a time, select with validation results, check calibration, and write the final comparison.

## TODO W4.1 Diagnose the initial MLP

Use the Week 3 learning curves to decide whether the initial model underfits, fits reasonably, or overfits. Compare training and validation loss, Accuracy, and Macro-F1, and cite the observed gap.

**Your diagnosis:**


In [ ]:
# TODO W4.2: choose controlled experiments relative to the initial MLP.
# Supply two small learning-rate variants. Use dropout for the required
# regularization variant so evaluation-mode behavior can be verified.
experiment_configs = [
    {
        "name": "learning_rate_variant_1",
        "learning_rate": None,
        "dropout": 0.0,
        "weight_decay": 0.0,
        "hypothesis": "",
    },
    {
        "name": "learning_rate_variant_2",
        "learning_rate": None,
        "dropout": 0.0,
        "weight_decay": 0.0,
        "hypothesis": "",
    },
    {
        "name": "dropout_regularization_variant",
        "learning_rate": 1e-3,
        "dropout": None,
        "weight_decay": 0.0,
        "hypothesis": "",
    },
]

if any(
    value is None
    for config in experiment_configs
    for key, value in config.items()
    if key in {"learning_rate", "dropout", "weight_decay"}
):
    raise NotImplementedError("TODO W4.2: choose the controlled experiment settings")
if any(not config["hypothesis"].strip() for config in experiment_configs):
    raise NotImplementedError("TODO W4.2: state a hypothesis for every experiment")

variant_runs = [
    train_mlp_experiment(
        name=config["name"],
        learning_rate=config["learning_rate"],
        dropout=config["dropout"],
        weight_decay=config["weight_decay"],
    )
    for config in experiment_configs
]

mlp_runs = {run["name"]: run for run in [initial_mlp_run, *variant_runs]}
for run in variant_runs:
    plot_training_history(run["history"], run["name"].replace("_", " ").title())


In [ ]:
def summarize_mlp_runs(mlp_runs: dict) -> pd.DataFrame:
    '''TODO W4.3a: create one row per run with settings, metrics, and gaps.'''
    raise NotImplementedError("TODO W4.3a: summarize every controlled experiment")


controlled_experiments = summarize_mlp_runs(mlp_runs)

# TODO W4.3b: select the final MLP using validation Macro-F1 only.
# Save final_mlp_name, final_mlp_run, and final_mlp.
final_mlp_name = None
final_mlp_run = None
final_mlp = None
raise NotImplementedError("TODO W4.3b: freeze a validation-selected final MLP")


In [ ]:
def top_label_calibration(logits: torch.Tensor, targets: torch.Tensor, n_bins: int = 10):
    '''TODO W4.4a: manually compute multiclass top-label reliability bins and ECE.'''
    # Validate logits [N, C] and targets [N].
    # Convert logits to probabilities with softmax.
    # For every example, retain top confidence and whether the top class is correct.
    # Bin confidences, then record count, mean confidence, empirical accuracy,
    # absolute gap, and bin weight. Store ECE in DataFrame.attrs.
    raise NotImplementedError("TODO W4.4a: implement top-label calibration")


# TODO W4.4b: call top_label_calibration on final validation logits, plot a
# reliability diagram against the y=x line, and report ECE.
calibration_summary = None
calibration_ece = None
raise NotImplementedError("TODO W4.4b: create the validation reliability diagram")


In [ ]:
# TODO W4.5: Build the final comparison table with these rows:
# - Trivial baseline
# - Locked Logistic Regression
# - Initial MLP
# - Final MLP
# Include train/validation Accuracy, train/validation Macro-F1, and notes.
model_comparison = None
raise NotImplementedError("TODO W4.5: create the final model-comparison table")


In [ ]:
def final_test_evaluation(selected_model, test_loader, loss_fn, device=DEVICE):
    if not RUN_FINAL_TEST:
        raise RuntimeError("Final test evaluation is disabled.")
    if not MODEL_SELECTION_FROZEN:
        raise RuntimeError("Freeze the model and settings before test evaluation.")
    return evaluate_model(selected_model, test_loader, loss_fn, device=device)


# TODO W4.6 OPTIONAL: Only after every choice is frozen and your instructor
# approves, set both flags in CONFIG to True and rerun the setup cell. Do not
# tune anything after viewing the result.
if RUN_FINAL_TEST and MODEL_SELECTION_FROZEN:
    print("Model selection frozen. Running the official held-out test exactly once.")
    test_loader = make_loader(x_scaled, y, test_idx, CONFIG["batch_size"], False, SEED)
    held_out_test_result = final_test_evaluation(
        final_mlp,
        test_loader,
        final_mlp_run["loss_fn"],
        DEVICE,
    )
    held_out_test_metrics = pd.Series({
        "model": final_mlp_name,
        "test_loss": held_out_test_result["loss"],
        "test_accuracy": held_out_test_result["accuracy"],
        "test_macro_f1": held_out_test_result["macro_f1"],
    }, name="held_out_test")
    display(held_out_test_metrics)
else:
    print("Final test remains disabled until model selection is frozen and approved.")


## TODO W4.7 Final interpretation and report

Write a short report that answers all of the following with values from your tables and plots:

- What did you try, and why?
- Which changes improved validation performance, and which did not?
- Did the initial and final MLP underfit, fit reasonably, or overfit?
- Did the MLP outperform Logistic Regression, and by how much?
- Was the added model complexity useful?
- Which model had the larger generalization gap?
- What did calibration reveal about confidence quality?
- What did you learn about optimization and regularization?

**Final report:**


# Debugging responses

Complete both checkpoint write-ups after repairing both bugs. For each checkpoint, record the observed behavior, the violated workflow rule, the smallest repair, and why the repair works.

## Checkpoint 1 Learned preprocessing

**Observed behavior:**

**Violated invariant:**

**Smallest repair:**

**Why the repair works:**

## Checkpoint 2 Evaluation behavior

**Observed behavior:**

**Violated invariant:**

**Smallest repair:**

**Why the repair works:**
